<a href="https://colab.research.google.com/github/isha0902/Deep_learning/blob/main/DLG_Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
from sklearn.datasets import make_moons

# ---------------- DATA ----------------
X, y = make_moons(
    n_samples=400,
    noise=0.20,
    random_state=1
)

y = y.reshape(-1, 1).astype(float)

# Standardize X
X = (X - X.mean(0)) / X.std(0)


# ---------------- ACTIVATION FUNCTIONS ----------------

def relu(z):
    return np.maximum(0, z)


def drelu(z):
    return (z > 0).astype(float)


def sig(z):
    return 1 / (1 + np.exp(-z))


# ---------------- INITIALIZATION ----------------

def init(sizes, seed=0):
    rng = np.random.default_rng(seed)
    P = []

    for a, b in zip(sizes[:-1], sizes[1:]):
        W = rng.normal(
            0,
            np.sqrt(2 / a),
            size=(a, b)
        )
        P.append([W, np.zeros((1, b))])

    return P


# ---------------- GRADIENT CALCULATION ----------------

def grads(P, Xb, yb):
    # Forward pass
    acts = [Xb]
    zs = []

    a = Xb

    for i, (W, b) in enumerate(P):
        z = a @ W + b
        zs.append(z)

        if i == len(P) - 1:
            a = sig(z)
        else:
            a = relu(z)

        acts.append(a)

    # Backward pass
    m = Xb.shape[0]

    g = [None] * len(P)

    # Sigmoid + Binary Cross Entropy
    dz = (acts[-1] - yb) / m

    for i in reversed(range(len(P))):

        # Gradient of weights
        g[i] = [
            acts[i].T @ dz,
            dz.sum(0, keepdims=True)
        ]

        # Move to previous layer
        if i > 0:
            dz = (dz @ P[i][0].T) * drelu(zs[i - 1])

    return g


# ---------------- ACCURACY ----------------

def accuracy(P, X, y):
    a = X

    for i, (W, b) in enumerate(P):
        if i == len(P) - 1:
            a = sig(a @ W + b)
        else:
            a = relu(a @ W + b)

    return np.mean((a >= 0.5) == (y >= 0.5))


# ---------------- VALUES ----------------

sizes, lr, EPOCHS = [2, 16, 16, 1], 0.5, 200


# ======================================================
# (A) BATCH GRADIENT DESCENT
# ALL DATA → ONE UPDATE PER EPOCH
# ======================================================

P = init(sizes, seed=0)

for epoch in range(EPOCHS):

    # Use the complete dataset
    g = grads(P, X, y)

    # Update weights and biases
    for k in range(len(P)):
        P[k][0] -= lr * g[k][0]
        P[k][1] -= lr * g[k][1]

print(
    "Batch GD accuracy:",
    accuracy(P, X, y)
)


# ======================================================
# (B) STOCHASTIC / MINI-BATCH GRADIENT DESCENT
# MINI-BATCH SIZE = 16
# ======================================================

P = init(sizes, seed=0)

B = 16
idx = np.arange(len(X))

for epoch in range(EPOCHS):

    # Shuffle data every epoch
    np.random.shuffle(idx)

    # Process mini-batches
    for s in range(0, len(X), B):

        b = idx[s:s+B]

        # Select mini-batch
        Xb = X[b]
        yb = y[b]

        # Calculate gradients
        g = grads(P, Xb, yb)

        # Update weights and biases
        for k in range(len(P)):
            P[k][0] -= lr * g[k][0]
            P[k][1] -= lr * g[k][1]


print(
    "SGD accuracy:",
    accuracy(P, X, y)
)


Batch GD accuracy: 0.965
SGD accuracy: 0.975
